In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

print("--- Starting Gold Layer: `dim_site` (SCD Type 2) ---")

# 1. Read current silver resolved records as source
df_silver = spark.read.table("inlap.silver.sites_resolved")

required_columns = {
    "resolved_glid",
    "resolved_baseglid",
    "street_address",
    "city",
    "state",
    "zip",
    "latitude",
    "longitude",
    "network_type",
    "status",
    "registry_confidence_score",
}
missing_columns = sorted(required_columns - set(df_silver.columns))
if missing_columns:
    raise ValueError(f"sites_resolved is missing required columns: {missing_columns}")

# Prepare source dataframe for SCD Type 2 fields using the current silver schema
# resolved_glid is the stable site-level key produced by the silver consolidation step.
df_incoming = (
    df_silver.select(
        F.col("resolved_glid").alias("master_site_id"),
        F.col("resolved_glid").alias("site_id"),
        F.col("resolved_glid"),
        F.col("resolved_baseglid"),
        F.col("street_address"),
        F.col("city"),
        F.col("state"),
        F.col("zip"),
        F.col("latitude"),
        F.col("longitude"),
        F.col("network_type"),
        F.col("status"),
        F.col("registry_confidence_score"),
        F.current_timestamp().alias("effective_start_date"),
    )
    .withColumn("effective_end_date", F.lit(None).cast("timestamp"))
    .withColumn("is_current", F.lit(True))
)

# 2. Check if gold dim_site table exists; if not, create initial baseline
table_name = "inlap.gold.dim_site"
spark.sql("CREATE DATABASE IF NOT EXISTS inlap.gold")

# Use standard Spark catalog check instead of internal _jcatalog
if not spark.catalog.tableExists(table_name):
    # Initial load: write all records with active flags
    (
        df_incoming.withColumn("surrogate_key", F.expr("uuid()"))
        .write.format("delta")
        .mode("overwrite")
        .saveAsTable(table_name)
    )
    print("Created `inlap.gold.dim_site` initial baseline.")
    display(spark.read.table(table_name).orderBy(F.col("master_site_id")).limit(5))
else:
    # SCD Type 2 MERGE operation using Delta Lake
    deltaTable = DeltaTable.forName(spark, table_name)
    print("Executing Delta MERGE for SCD Type 2 updates...")

    # Delta merge logic for handling updates/history tracking
    # (Matches on master_site_id for active records where attributes changed)